# Jornada completa — Planilha Turmas (DEMO)

Execute **Restart Kernel and Run All**. Dados sintéticos reproduzíveis (seed 42), sem fonte institucional. Métricas demonstram integração, não qualidade real. O grão é curso/período/disciplina/ciclo. A demanda histórica do período é atribuída às disciplinas pela regra provisória da POC; não é decisão funcional aprovada do MER.

O notebook abre SQL com usuário de leitura; usa MLflow remoto em proxy, recarrega uma versão explícita e grava uma proposta DRAFT em transação. Não promove modelos. Cada Run All cria uma nova execução; repetir apenas a persistência com o mesmo conteúdo é idempotente.


## 1. Configurações e dependências

Testar SQL de origem/destino, versão do SDK/servidor, tracking real e upload/download pelo proxy antes de qualquer fit.

**Esperado/evidência:** Tabela PASS; se houver FAIL o notebook interrompe com diagnóstico, sem tracking local.


In [1]:
import os, uuid, time, json
from pathlib import Path
import pandas as pd
import numpy as np
import httpx
import mlflow
import mlflow.sklearn
from IPython.display import display, Markdown
from src.workbench.journey_config import Settings
from src.workbench.journey_steps import dependencies, require_dependencies, step
cfg = Settings.load()
execution_id = str(uuid.uuid4())
statuses = []
checks = dependencies(cfg, execution_id)
display(pd.DataFrame(checks))
require_dependencies(checks, cfg)
statuses.append(dict(stage="dependencies", status="PASS"))
display({"execution_id":execution_id, "mode":cfg.mode, "tracking":cfg.tracking})

2026/10/30 13:16:07 WARNING mlflow.utils.git_utils: Failed to import Git (the Git executable is probably not on your PATH), so Git SHA is not available. Error: Failed to initialize: Bad git executable.
The git executable must be specified in one of the following ways:
    - be included in your $PATH
    - be set via $GIT_PYTHON_GIT_EXECUTABLE
    - explicitly set via git.refresh(<full-path-to-git-executable>)

All git commands will error until this is rectified.

This initial message can be silenced or aggravated in the future by setting the
$GIT_PYTHON_REFRESH environment variable. Use one of the following values:
    - quiet|q|silence|s|silent|none|n|0: for no message or exception
    - warn|w|warning|log|l|1: for a warning message (logging level CRITICAL, displayed by default)
    - error|e|exception|raise|r|2: for a raised exception

Example:
    export GIT_PYTHON_REFRESH=quiet



🏃 View run respected-sponge-217 at: http://mlflow:5000/#/experiments/3/runs/a2c78074e2b442dfa020291f92ab0184
🧪 View experiment at: http://mlflow:5000/#/experiments/3


,dependency,status,attempts
0,source SQL,PASS,1
1,application SQL,PASS,1
2,MLflow PostgreSQL + proxy artifact roundtrip,PASS,1


{'execution_id': 'a741d0dd-d1ed-41eb-91d1-d5194c5fb692',
 'mode': 'DEMO',
 'tracking': 'http://mlflow:5000'}

## 2. Extração SQL versionada

A conexão é aberta por src.data.journey usando SOURCE_DATABASE_URL. Fonte DEMO: planejamento_db.planning.historical_enrollments e catálogo relacionado. O seed já foi carregado no bootstrap.

**Esperado/evidência:** Consulta parametrizada, snapshot/hash, filtros, contagens, tipos e amostra sem credenciais.


In [2]:
from src.data.journey import extract, prepare, FEATURES, TARGET, QUERY
with step("SQL extraction", execution_id, statuses):
    history, snapshot = extract(cfg)
    display(Markdown("```sql\n" + QUERY.read_text() + "\n```"))
    display({"source":"planejamento_db/planning/SYNTHETIC", "cycle":cfg.cycle,
        "course":cfg.course or "all", "snapshot":snapshot, "rows":len(history),
        "nulls":int(history.isna().sum().sum()), "duplicates":int(history.duplicated().sum())})
    display(history.dtypes.astype(str).rename("type").to_frame())
    display(history.head())

{"timestamp": "2026-10-30T13:16:08.070262+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "SQL extraction", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


```sql
-- journey-history-v1. Target: total students per period, replicated to its disciplines
-- under the provisional POC rule: every discipline receives the whole cohort.
SELECT h.course_id::text, h.period_id::text, pd.discipline_id::text,
       h.reference_year * 2 + h.reference_semester - 1 AS semester_index,
       h.reference_semester AS semester,
       h.regular_students + h.dependency_students + h.elective_students AS students
FROM planning.historical_enrollments h
JOIN planning.dataset_snapshots s ON s.id = h.snapshot_id
JOIN planning.academic_cycles c ON c.id = s.cycle_id
JOIN planning.period_disciplines pd ON pd.period_id = h.period_id
WHERE h.snapshot_id = CAST(:snapshot AS uuid) AND c.cycle_code = :cycle
  AND s.sealed_at IS NOT NULL
  AND (CAST(:course AS text) IS NULL OR h.course_id::text = :course)
  AND h.reference_year * 2 + h.reference_semester - 1 < c.target_year * 2 + c.target_semester - 1
ORDER BY semester_index, h.course_id, h.period_id, pd.discipline_id

```

{'source': 'planejamento_db/planning/SYNTHETIC',
 'cycle': '2026-1',
 'course': 'all',
 'snapshot': {'id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
  'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
  'target_index': 4052,
  'target_semester': 1,
  'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
  'query_sha256': 'defc10d1b6ea752600af271285c54107663e91db9df13914aea7e42613c83adf',
  'query_version': 'journey-history-v1'},
 'rows': 1050,
 'nulls': 0,
 'duplicates': 0}

,type
course_id,object
period_id,object
discipline_id,object
semester_index,int64
semester,int64
students,int64


,course_id,period_id,discipline_id,semester_index,semester,students
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,4040,1,86
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,4040,1,86
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,4040,1,86
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,4040,1,82
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,4040,1,82


{"timestamp": "2026-10-30T13:16:08.096823+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "SQL extraction", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.027}


## 3. Preparação temporal

Reservar o último semestre histórico para avaliação. O conjunto de proposta contém somente chaves acadêmicas e o próximo semestre, sem alvo. Transformações serão ajustadas exclusivamente no treino.

**Esperado/evidência:** Treino < avaliação < proposta no tempo.


In [3]:
with step("prepare", execution_id, statuses):
    train, evaluation, future = prepare(history, snapshot)
    display(pd.DataFrame([
        {"dataset":"train", "rows":len(train), "start":int(train.semester_index.min()), "end":int(train.semester_index.max())},
        {"dataset":"evaluation", "rows":len(evaluation), "start":int(evaluation.semester_index.min()), "end":int(evaluation.semester_index.max())},
        {"dataset":"proposal", "rows":len(future), "start":int(future.semester_index.min()), "end":int(future.semester_index.max())}]))

{"timestamp": "2026-10-30T13:16:08.102520+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "prepare", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


,dataset,rows,start,end
0,train,960,4040,4050
1,evaluation,90,4051,4051
2,proposal,90,4052,4052


{"timestamp": "2026-10-30T13:16:08.108017+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "prepare", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.005}


## 4. Modelo e métricas

Ridge(alpha=1, seed=42), one-hot de curso/período/disciplina e padronização temporal. Alvo: estudantes do período atribuídos à disciplina. MAE mede erro absoluto médio; RMSE penaliza mais os erros grandes. Ambas em estudantes.

**Esperado/evidência:** Pipeline instanciado e funções de avaliação definidas antes do fit.


In [4]:
from src.training.journey import estimator, evaluate, SEED
with step("model and metrics", execution_id, statuses):
    model = estimator()
    display({"features":FEATURES, "target":TARGET, "seed":SEED, "metrics":["MAE", "RMSE"]})
    display(model)

{"timestamp": "2026-10-30T13:16:08.373387+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "model and metrics", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'features': ['course_id',
  'period_id',
  'discipline_id',
  'semester_index',
  'semester'],
 'target': 'students',
 'seed': 42,
 'metrics': ['MAE', 'RMSE']}

Pipeline(steps=[('features',
                 ColumnTransformer(transformers=[('academic',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  ['course_id', 'period_id',
                                                   'discipline_id']),
                                                 ('time', StandardScaler(),
                                                  ['semester_index',
                                                   'semester'])])),
                ('regression', Ridge(random_state=42))])

{"timestamp": "2026-10-30T13:16:08.382467+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "model and metrics", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.009}


## 5. Experimento e execução

Criar/recuperar o experimento proxy e registrar identidade da extração, query, código, features e referência sintética. Cada etapa posterior encerra a run como FAILED se ocorrer erro.

**Esperado/evidência:** experiment_id/run_id e link clicável; CSV de referência e SQL enviados pelo proxy.


In [5]:
from src.mlops.journey import start, register
with step("tracking", execution_id, statuses):
    trace = start(cfg, execution_id, snapshot, history)
    display(trace)
    display(Markdown(f"[Abrir execução no MLflow]({trace['run_url']})"))

{"timestamp": "2026-10-30T13:16:08.460898+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "tracking", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'run_id': 'ad2530361ab14f7d87017c21d11958a1',
 'experiment_id': '4',
 'run_url': 'http://127.0.0.1:5000/#/experiments/4/runs/ad2530361ab14f7d87017c21d11958a1'}

[Abrir execução no MLflow](http://127.0.0.1:5000/#/experiments/4/runs/ad2530361ab14f7d87017c21d11958a1)

{"timestamp": "2026-10-30T13:16:08.629798+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "tracking", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.169}


## 6. Treinamento e avaliação

Executar fit/predict reais. O modelo persistido é exatamente o avaliado; não há retreino oculto com o conjunto de avaliação.

**Esperado/evidência:** MAE/RMSE calculados e registrados; métricas DEMO não comprovam qualidade institucional.


In [6]:
with step("fit and evaluate", execution_id, statuses):
    started = time.monotonic()
    model.fit(train[FEATURES], train[TARGET])
    predicted = model.predict(evaluation[FEATURES])
    metrics = evaluate(evaluation[TARGET], predicted)
    mlflow.log_metrics({**metrics, "training_seconds":time.monotonic()-started})
    mlflow.log_dict({"actual":evaluation[TARGET].tolist(), "predicted":predicted.tolist()}, "evaluation.json")
    display(metrics)

{"timestamp": "2026-10-30T13:16:08.635016+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "fit and evaluate", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'mae': 3.758630658273197, 'rmse': 4.9142272796917865}

{"timestamp": "2026-10-30T13:16:08.672844+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "fit and evaluate", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.038}


## 7. Persistência do modelo e Registry

O SDK envia modelo, transformações, assinatura e exemplo ao MLflow; somente o servidor grava o RustFS. A versão será selecionada explicitamente para teste DRAFT, sem alias champion.

**Esperado/evidência:** Nome, versão, URI e run; objeto treinado removido da memória.


In [7]:
with step("register", execution_id, statuses):
    trace = register(cfg, model, train[FEATURES].head(3), trace, execution_id)
    mlflow.end_run(status="FINISHED")
    display(trace)
    del model

{"timestamp": "2026-10-30T13:16:08.679151+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "register", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


/opt/conda/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/10/30 13:16:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during de

2026/10/30 13:16:09 INFO mlflow.models.model: Found the following environment variables used during model inference: [API_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


Registered model 'planilha-turmas-demo' already exists. Creating a new version of this model...
2026/10/30 13:16:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 60 seconds for model version to finish creation. Model name: planilha-turmas-demo, version 3


Created version '3' of model 'planilha-turmas-demo'.


🏃 View run powerful-ox-569 at: http://mlflow:5000/#/experiments/4/runs/ad2530361ab14f7d87017c21d11958a1
🧪 View experiment at: http://mlflow:5000/#/experiments/4


{'run_id': 'ad2530361ab14f7d87017c21d11958a1',
 'experiment_id': '4',
 'run_url': 'http://127.0.0.1:5000/#/experiments/4/runs/ad2530361ab14f7d87017c21d11958a1',
 'model_name': 'planilha-turmas-demo',
 'model_version': '3',
 'model_uri': 'models:/planilha-turmas-demo/3',
 'artifact_uri': 'models:/m-1717e4e5a7d540de8ecd8aed5e40a049'}

{"timestamp": "2026-10-30T13:16:09.401953+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "register", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.723}


## 8. Recarregar e inferir no serviço existente

Baixar a versão registrada e executar predict no conjunto futuro. Chamar também o serviço inference: ele carrega a mesma versão e verifica run/execução/snapshot. Registro não significa promoção operacional.

**Esperado/evidência:** Download real do artefato; serviço carregou versão igual e produziu previsões iguais.


In [8]:
from src.inference.journey import load_version
from src.common.settings import required
with step("reload and inference", execution_id, statuses):
    restored, registered = load_version(trace["model_name"], trace["model_version"], trace["run_id"], execution_id)
    predictions = restored.predict(future[FEATURES])
    response = httpx.post(cfg.inference + "/internal/draft/predict", timeout=120,
        headers={"Authorization":"Bearer " + required("INFERENCE_TOKEN"), "X-Correlation-ID":execution_id},
        json={"execution_id":execution_id, "snapshot_id":snapshot["id"], "model_name":trace["model_name"],
              "model_version":trace["model_version"], "run_id":trace["run_id"], "records":future[FEATURES].to_dict("records")})
    response.raise_for_status()
    inference = response.json()
    assert inference["model_version"] == trace["model_version"] and inference["run_id"] == trace["run_id"]
    np.testing.assert_allclose(predictions, inference["predictions"], rtol=1e-10)
    client = mlflow.MlflowClient()
    client.log_metric(trace["run_id"], "inference_rows", len(predictions))
    client.log_metric(trace["run_id"], "inference_seconds", inference["duration_seconds"])
    display({k:v for k,v in inference.items() if k != "predictions"})
    display(future.assign(forecast_raw=predictions).head())

{"timestamp": "2026-10-30T13:16:09.410773+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "reload and inference", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'execution_id': 'a741d0dd-d1ed-41eb-91d1-d5194c5fb692',
 'run_id': 'ad2530361ab14f7d87017c21d11958a1',
 'model_name': 'planilha-turmas-demo',
 'model_version': '3',
 'loaded': True,
 'operationally_promoted': False,
 'duration_seconds': 0.16026599899998928}

,course_id,period_id,discipline_id,semester_index,semester,forecast_raw
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,4052,1,102.089955
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,4052,1,102.089955
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,4052,1,102.089955
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,4052,1,103.245511
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,4052,1,103.245511


{"timestamp": "2026-10-30T13:16:09.757744+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "reload and inference", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.347}


## 9. Compor Planilha Turmas

Regra provisória POC: estudantes propostos = max(0, ceil(previsão)); turmas = ceil(estudantes/capacidade). Capacidade demonstrativa = 40. Zero demanda gera zero turmas. Sem ajustes humanos nesta execução.

**Esperado/evidência:** Valor previsto, estudantes propostos, turmas e ajuste humano separados.


In [9]:
from src.persistence.journey import compose, persist, RULE
with step("compose proposal", execution_id, statuses):
    capacity = 40
    lines = compose(future, predictions, capacity)
    display({"rule":RULE, "capacity":capacity, "state":"DRAFT"})
    display(pd.DataFrame(lines).head())

{"timestamp": "2026-10-30T13:16:09.766587+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "compose proposal", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'rule': 'POC-whole-cohort-ceil-capacity-v1', 'capacity': 40, 'state': 'DRAFT'}

,course_id,period_id,discipline_id,forecast_raw,proposed_students,proposed_groups,human_adjustment
0,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,102.089955,103,3,None
1,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,102.089955,103,3,None
2,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,102.089955,103,3,None
3,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,103.245511,104,3,None
4,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,103.245511,104,3,None


{"timestamp": "2026-10-30T13:16:09.774457+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "compose proposal", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.008}


## 10. Persistir e verificar idempotência

Gravar cabeçalho e todas as linhas numa transação PostgreSQL em planejamento_db.planning.proposal_*. Repetição idêntica retorna a geração existente; conteúdo diferente com o mesmo UUID é rejeitado. Novo Run All preserva as versões anteriores.

**Esperado/evidência:** Geração DRAFT persistida com vínculos SQL/run/modelo/regra; repetição sem duplicações.


In [10]:
with step("persist and idempotency", execution_id, statuses):
    generation = persist(cfg, execution_id, snapshot, trace, lines, capacity)
    repeated = persist(cfg, execution_id, snapshot, trace, lines, capacity)
    assert generation["execution_id"] == repeated["execution_id"]
    assert generation["row_count"] == len(lines)
    display(generation)

{"timestamp": "2026-10-30T13:16:09.802397+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "persist and idempotency", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'execution_id': UUID('a741d0dd-d1ed-41eb-91d1-d5194c5fb692'),
 'correlation_id': UUID('a741d0dd-d1ed-41eb-91d1-d5194c5fb692'),
 'cycle_id': UUID('0fa6938d-2743-56ae-a057-dffd2274d90a'),
 'snapshot_id': UUID('fb931e1e-43d0-5ff8-938c-0d2ba4a68794'),
 'course_scope': None,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'payload_sha256': '00bd16eb962ec0763f4881653921c33f3a0bca799a6d8cacb72975506dd567ea',
 'run_id': 'ad2530361ab14f7d87017c21d11958a1',
 'experiment_id': '4',
 'model_name': 'planilha-turmas-demo',
 'model_version': '3',
 'artifact_uri': 'models:/m-1717e4e5a7d540de8ecd8aed5e40a049',
 'rule_version': 'POC-whole-cohort-ceil-capacity-v1',
 'capacity': 40,
 'mode': 'DEMO',
 'status': 'DRAFT',
 'row_count': 90,
 'generated_at': datetime.datetime(2026, 10, 30, 13, 16, 9, 815435, tzinfo=zoneinfo.ZoneInfo(key='Etc/UTC'))}

{"timestamp": "2026-10-30T13:16:09.841922+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "persist and idempotency", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.04}


## 11. Consultar API e reconciliar

A API consulta PostgreSQL sem treinamento. Comparar todas as linhas e rastreabilidade com a proposta escrita pelo notebook.

**Esperado/evidência:** Mesmos IDs, quantidades e total no SQL e na API; run ligada à geração.


In [11]:
with step("API reconciliation", execution_id, statuses):
    headers = {"Authorization":"Bearer " + required("API_TOKEN"), "X-Correlation-ID":execution_id}
    def get_api(path):
        r = httpx.get(cfg.api + path, headers=headers, timeout=cfg.timeout)
        r.raise_for_status()
        return r.json()
    api_generation = get_api(f"/v1/proposals/{execution_id}")
    api_lines = get_api(f"/v1/proposals/{execution_id}/lines?limit=500")
    assert api_generation["run_id"] == trace["run_id"]
    assert api_generation["model_version"] == trace["model_version"]
    assert len(api_lines) == len(lines) == api_generation["row_count"]
    expected = {tuple(r[k] for k in ["course_id","period_id","discipline_id"]):r for r in lines}
    for row in api_lines:
        original = expected[tuple(row[k] for k in ["course_id","period_id","discipline_id"])]
        assert row["proposed_students"] == original["proposed_students"]
        assert row["proposed_groups"] == original["proposed_groups"]
        assert row["forecast_raw"] == original["forecast_raw"]
    display(api_generation)
    display(pd.DataFrame(api_lines).head())

{"timestamp": "2026-10-30T13:16:09.850636+00:00", "service": "journey", "level": "INFO", "message": "stage started", "stage": "API reconciliation", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692"}


{'execution_id': 'a741d0dd-d1ed-41eb-91d1-d5194c5fb692',
 'correlation_id': 'a741d0dd-d1ed-41eb-91d1-d5194c5fb692',
 'cycle_id': '0fa6938d-2743-56ae-a057-dffd2274d90a',
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'course_scope': None,
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'payload_sha256': '00bd16eb962ec0763f4881653921c33f3a0bca799a6d8cacb72975506dd567ea',
 'run_id': 'ad2530361ab14f7d87017c21d11958a1',
 'experiment_id': '4',
 'model_name': 'planilha-turmas-demo',
 'model_version': '3',
 'artifact_uri': 'models:/m-1717e4e5a7d540de8ecd8aed5e40a049',
 'rule_version': 'POC-whole-cohort-ceil-capacity-v1',
 'capacity': 40,
 'mode': 'DEMO',
 'status': 'DRAFT',
 'row_count': 90,
 'generated_at': '2026-10-30T13:16:09.815435+00:00'}

,execution_id,course_id,period_id,discipline_id,forecast_raw,proposed_students,proposed_groups,human_adjustment
0,a741d0dd-d1ed-41eb-91d1-d5194c5fb692,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,172d26b5-b285-5ec4-8971-f3f1cac5befb,102.089955,103,3,None
1,a741d0dd-d1ed-41eb-91d1-d5194c5fb692,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,3c7c40f4-7b48-5e65-ad02-fdf7381ae7a3,102.089955,103,3,None
2,a741d0dd-d1ed-41eb-91d1-d5194c5fb692,0169f756-26d1-5ad6-acd4-4e6b0e319939,39c82c86-d06e-5f26-9712-a92b4018084e,5bb84918-2141-55b8-aab6-01257acfa73d,102.089955,103,3,None
3,a741d0dd-d1ed-41eb-91d1-d5194c5fb692,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,53abed31-c5c0-504e-a7a9-f89f13a31fb3,103.245511,104,3,None
4,a741d0dd-d1ed-41eb-91d1-d5194c5fb692,0169f756-26d1-5ad6-acd4-4e6b0e319939,519ead55-4294-52e1-b326-550e4be54cd0,8098c200-9822-56dc-a701-71f67f356b82,103.245511,104,3,None


{"timestamp": "2026-10-30T13:16:09.886167+00:00", "service": "journey", "level": "INFO", "message": "stage completed", "stage": "API reconciliation", "correlation_id": "a741d0dd-d1ed-41eb-91d1-d5194c5fb692", "duration_seconds": 0.036}


## 12. Evidências finais

Salvar resumo reproduzível da execução no workspace e no MLflow. Consultar dashboard pelo correlation_id.

**Esperado/evidência:** Todas as etapas PASS, evidência local e no Artifact Store, links sem segredos.


In [12]:
summary = {"execution_id":execution_id, "mode":cfg.mode, "sql_rows":len(history), "metrics":metrics,
    **trace, "proposal_id":execution_id, "persisted_rows":len(lines), "api_rows":len(api_lines),
    "snapshot_id":snapshot["id"], "dataset_sha256":snapshot["dataset_sha256"], "stages":statuses}
reports = Path("/workspace/reports")
reports.mkdir(parents=True, exist_ok=True)
report = reports / f"journey-{execution_id}.json"
report.write_text(json.dumps(summary, indent=2), encoding="utf-8")
mlflow.MlflowClient().log_artifact(trace["run_id"], str(report), "evidence")
display(pd.DataFrame(statuses))
display(summary)
display(Markdown(f"[Run MLflow]({trace['run_url']}) · [API](http://127.0.0.1:8000/docs) · [Dashboard](http://127.0.0.1:8088) · [RustFS](http://127.0.0.1:9001)"))

,stage,status,seconds
0,dependencies,PASS,NaN
1,SQL extraction,PASS,0.027
2,prepare,PASS,0.005
3,model and metrics,PASS,0.009
4,tracking,PASS,0.169
5,fit and evaluate,PASS,0.038
6,register,PASS,0.723
7,reload and inference,PASS,0.347
8,compose proposal,PASS,0.008
9,persist and idempotency,PASS,0.040


{'execution_id': 'a741d0dd-d1ed-41eb-91d1-d5194c5fb692',
 'mode': 'DEMO',
 'sql_rows': 1050,
 'metrics': {'mae': 3.758630658273197, 'rmse': 4.9142272796917865},
 'run_id': 'ad2530361ab14f7d87017c21d11958a1',
 'experiment_id': '4',
 'run_url': 'http://127.0.0.1:5000/#/experiments/4/runs/ad2530361ab14f7d87017c21d11958a1',
 'model_name': 'planilha-turmas-demo',
 'model_version': '3',
 'model_uri': 'models:/planilha-turmas-demo/3',
 'artifact_uri': 'models:/m-1717e4e5a7d540de8ecd8aed5e40a049',
 'proposal_id': 'a741d0dd-d1ed-41eb-91d1-d5194c5fb692',
 'persisted_rows': 90,
 'api_rows': 90,
 'snapshot_id': 'fb931e1e-43d0-5ff8-938c-0d2ba4a68794',
 'dataset_sha256': '5fc39eeeead59aa48813182e6a93d2308a4a9243655501f17d09a981c6c66a56',
 'stages': [{'stage': 'dependencies', 'status': 'PASS'},
  {'stage': 'SQL extraction', 'status': 'PASS', 'seconds': 0.027},
  {'stage': 'prepare', 'status': 'PASS', 'seconds': 0.005},
  {'stage': 'model and metrics', 'status': 'PASS', 'seconds': 0.009},
  {'stage': 

[Run MLflow](http://127.0.0.1:5000/#/experiments/4/runs/ad2530361ab14f7d87017c21d11958a1) · [API](http://127.0.0.1:8000/docs) · [Dashboard](http://127.0.0.1:8088) · [RustFS](http://127.0.0.1:9001)